# FT-900 全训练期 Exhibit 7 原件行审核

在已验证的 144 份原始 PDF 字节上，只读抽取每月半导体出口行和报表首页发布标识。版式例外保持 unknown，不补值，不使用 SOXX 收益。

In [ ]:
from pathlib import Path
import hashlib,json,re,fitz
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023'
plan=json.loads((card/'full_training_row_audit_plan.json').read_text());parent=json.loads((card/plan['parent_source_result']).read_text());readback=json.loads((card/plan['parent_readback']).read_text())
assert parent['target_months']==plan['expected_months']==144 and readback['monthly_PDF_hashes_verified']==144
manifest=json.loads((card/'full_training_url_manifest.json').read_text())
receipts=[json.loads(x) for x in (card/plan['source_receipts']).read_text().splitlines()]
finished={x['month']:x for x in receipts if x['event'] in ('completed','reused')}
assert len(finished)==144 and set(finished)==set(manifest['month_urls'])
rows=[]
for month in sorted(finished):
    item=finished[month];path=root/item['local_path'];payload=path.read_bytes()
    assert len(payload)==item['bytes'] and hashlib.sha256(payload).hexdigest()==item['sha256']
    document=fitz.open(stream=payload,filetype='pdf');first=' '.join(document[0].get_text().split())
    header=re.search(r'FOR\s+(?:IMMEDIATE\s+)?RELEASE.{0,170}?(?:19|20)\d{2}',first,re.I)
    candidates=[]
    for page_no,page in enumerate(document,1):
        text=page.get_text();normalized=' '.join(text.split())
        if 'Exhibit 7.' not in normalized or 'Exports of Goods by End-Use Category and Commodity' not in normalized:continue
        lines=[x.strip() for x in text.splitlines()]
        for idx,line in enumerate(lines):
            if line=='Semiconductors':
                next_lines=lines[idx+1:idx+7]
                candidates.append({'page':page_no,'tokens_after_label':next_lines})
    result={'reference_month':month,'source_pdf_sha256':item['sha256'],'source_bytes':item['bytes'],'source_url':item['url'],'printed_release_header':header.group(0) if header else None,'row_candidate_count':len(candidates),'row_candidates':candidates,'status':'needs_review'}
    if len(candidates)==1:
        tokens=candidates[0]['tokens_after_label']
        if len(tokens)>=3 and all(re.fullmatch(r'-?[\d,]+',t) for t in tokens[:3]):
            values=[int(t.replace(',','')) for t in tokens[:3]]
            result.update(status='unique_numeric_Exhibit7_row',page=candidates[0]['page'],current_month_million_usd=values[0],prior_month_same_release_million_usd=values[1],printed_change_million_usd=values[2],printed_change_minus_rounded_arithmetic=values[2]-(values[0]-values[1]))
    rows.append(result)
counts={'unique_numeric_Exhibit7_rows':sum(x['status']=='unique_numeric_Exhibit7_row' for x in rows),'release_headers_found':sum(bool(x['printed_release_header']) for x in rows),'exceptions':[x['reference_month'] for x in rows if x['status']!='unique_numeric_Exhibit7_row']}
result={'status':'full_row_audit_complete' if counts['unique_numeric_Exhibit7_rows']==144 else 'full_row_audit_exceptions_preserved','source_months':144,**counts,'historical_first_public_bytes_qualified':False,'classification_continuity_qualified':False,'Qlib_factor_rows':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'rows':rows}
(card/'full_training_row_audit.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:v for k,v in result.items() if k in ('status','source_months','unique_numeric_Exhibit7_rows','release_headers_found','exceptions')})

## 结论边界

逐月行存在和数值审核只决定源字段覆盖。2012 分类调整、历史首次公开字节和 Stage1 增量信息仍需单独核对。无模型、账户或 SOXX 超额结论。